# EXP-2026-009 — Q5-D negative-control null 아티팩트 복구 (미실행 템플릿)

이 노트북은 **포장 복구**다. 과학 실험이 아니다.

- beat join 재실행 없음 · null 재실행 없음 · `J` 값 계산 없음
- 기존 Drive bundle·shard **수정·삭제·덮어쓰기 없음**
- 새 corrective 폴더 하나만 만들고, 거기에 **`BUNDLE_FILES` 12개 외에는 아무것도 넣지 않는다**

명세: `experiments/specs/EXP-2026-009-q5d-null-artifact-repair.md`

**현재 실행 승인은 없다.** `EXECUTION_APPROVAL_RECORD['granted']` 가 `False` 이고
`APPROVED_IMPLEMENTATION_COMMIT` 도 비어 있으므로 실행 셀은 거부된다.

## 이 노트북이 인증을 하지 않는 이유

`auth.authenticate_user()` 나 `build('drive','v3')` 를 셀에서 직접 부르지 않는다.
그렇게 하면 terminal guard 가 **보지 못한** credential 이 만들어져, guard 가
멀쩡히 있는 채로 무력해진다. 인증은 `run_repair()` 안, guard **아래**에서
일어나고, 이 노트북은 authenticator 와 service factory 를 넘겨줄 뿐이다.

## 실패 시 무슨 일이 일어나는가 (읽고 실행하라)

- **target 생성 전 실패** → 폴더가 아예 만들어지지 않는다.
- **target 생성 후 실패** → 그 폴더를 **그 자리에 그대로 보존**한다. 경로와 파일
  목록을 보고하고 `REPAIR_INCOMPLETE_TARGET_PRESERVED` 로 표시한다.
  **COMMITTED 도 accepted 도 아니고 등록 대상도 아니다.**
- **12파일을 다 쓰고 folder ID 만 안 잡힐 때** → `REPAIR_OUTPUT_FOLDER_ID_UNRESOLVED`.
  출력은 보존되고, **두 번째 폴더를 만들지 않는다.** 마지막 셀의 read-only
  reconciliation 으로 나중에 ID 만 다시 잡는다.
- 이 모듈은 **어떤 경우에도 삭제·덮어쓰기·rename 을 하지 않는다.**
- **재시도는 반드시 새 고유 경로**로 한다.

In [ ]:
# 1. PINNED CHECKOUT — 실행은 움직이는 branch 가 아니라 정확한 커밋에 고정한다.
#    다만 여기 적는 40-hex 는 "승인"이 아니라 **관측**이다. 승인된 구현 identity 는
#    모듈의 APPROVED_IMPLEMENTATION_COMMIT / APPROVED_ARTIFACT_DIGESTS 에 있고,
#    그것은 이 커밋보다 **나중 커밋**이 기록한다(자기참조 불가).
import os, sys, json, subprocess

CHECKOUT_COMMIT = ''   # 체크아웃할 커밋(40-hex). 비우면 거부.
REPO_URL = 'https://github.com/ehdbddl06001-ui/my-github-test.git'
CLONE_TO = '/content/repo'
NEEDED = ('q5d_order_preserving_beat_join.py', 'q5d_null_artifact_repair.py')

if not CHECKOUT_COMMIT or len(CHECKOUT_COMMIT) != 40:
    raise RuntimeError(
        '체크아웃할 정확한 커밋 SHA(40-hex)를 CHECKOUT_COMMIT 에 적어야 한다. '
        'branch 이름은 시간에 따라 움직이므로 고정이 아니다.')


def _is_repo(path):
    if not path:
        return False
    here = os.path.join(path, 'mit-bih')
    return all(os.path.isfile(os.path.join(here, n)) for n in NEEDED)


if not _is_repo(CLONE_TO):
    print('clone:', REPO_URL, '→', CLONE_TO)
    _r = subprocess.run(['git', 'clone', REPO_URL, CLONE_TO],
                        capture_output=True, text=True)
    print((_r.stdout or '').strip() or (_r.stderr or '').strip())

_r = subprocess.run(['git', '-C', CLONE_TO, 'checkout', '--detach',
                     CHECKOUT_COMMIT], capture_output=True, text=True)
print((_r.stdout or '').strip() or (_r.stderr or '').strip())
if _r.returncode != 0:
    raise RuntimeError(f'{CHECKOUT_COMMIT} 체크아웃 실패.')

# EXECUTION_HEAD 는 git 에서 **실측**한다. 위에 적은 값을 그대로 믿지 않는다.
EXECUTION_HEAD = subprocess.run(['git', '-C', CLONE_TO, 'rev-parse', 'HEAD'],
                                capture_output=True, text=True).stdout.strip()
if EXECUTION_HEAD != CHECKOUT_COMMIT:
    raise RuntimeError(f'HEAD 가 {EXECUTION_HEAD} 로 적어둔 값과 다르다.')

_dirty = subprocess.run(['git', '-C', CLONE_TO, 'status', '--porcelain'],
                        capture_output=True, text=True).stdout.strip()
if _dirty:
    raise RuntimeError(f'작업 트리가 깨끗하지 않다 — 커밋 고정이 무의미해진다:\n{_dirty}')

REPO = CLONE_TO
_MITBIH = os.path.join(REPO, 'mit-bih')
if _MITBIH not in sys.path:
    sys.path.insert(0, _MITBIH)

import q5d_order_preserving_beat_join as BJ
import q5d_null_artifact_repair as R

MISSING = [n for n in R.module_capabilities() if not hasattr(R, n)]
assert not MISSING, f'stale repair clone, missing {MISSING}'

print('REPO          :', REPO)
print('EXECUTION_HEAD:', EXECUTION_HEAD, '(git 실측)')
print()
print(R.design_card())

In [ ]:
# 2. 승인된 구현과 실제 실행 코드가 같은가 — 커밋을 아는 것과 디스크의 파일이
#    그 커밋의 것임을 아는 것은 다르다. 등록 identity 는 LF 정규화 SHA 이고
#    raw byte SHA 도 함께 본다.
print('approved implementation commit:', R.APPROVED_IMPLEMENTATION_COMMIT)
print('approved artifact digests     :')
for _k, _v in R.APPROVED_ARTIFACT_DIGESTS.items():
    print(f'  {_k:28s} {_v}')
print()
_science = R.module_science_digest(os.path.join(REPO, R.MODULE_PATH))
print('module science digest (승인 블록 제외):',
      _science['module_science_lf_sha256'])
print('  ', _science['convention'])
print()
for _label, _e in R.artifact_identities(REPO).items():
    print(f"{_label:9s} {_e['path']}")
    print(f"          LF  {_e['lf_normalized_sha256']}")
    print(f"          raw {_e['raw_sha256']}  (CRLF: {_e['had_crlf']})")

print()
FROZEN = R.assert_frozen_q5d_unchanged()
print('frozen Q5-D LF  :', FROZEN['lf_normalized_sha256'], '← 등록 기준')
print('frozen Q5-D raw :', FROZEN['raw_sha256'], '(CRLF:', FROZEN['had_crlf'], ')')
print('rule fingerprint:', FROZEN['rule_fingerprint'])
print()
print(R.NEWLINE_CONVENTION)
print()
print(R.MEMBER_NAMING_NOTE)

In [ ]:
# 3. 합성 fixture 검증 — 실제 자산을 열기 전 가장 싼 관문.
#    실패하면 stderr 와 종료 코드를 보이고 멈춘다(조용한 실패 금지).
_res = subprocess.run(
    [sys.executable, os.path.join(REPO, 'mit-bih',
                                  'test_q5d_null_artifact_repair.py')],
    capture_output=True, text=True)
print((_res.stdout or '').strip() or '(stdout 없음)')
if _res.returncode != 0:
    print('--- stderr ---')
    print((_res.stderr or '').strip() or '(stderr 없음)')
    raise RuntimeError(
        f'합성 fixture 실패 (exit {_res.returncode}). 실행 셀을 누르지 마라.')

In [ ]:
# 4. 경로와 folder ID — 폴더는 **ID 로만** 지정한다. 이름이 같은 다른 폴더는
#    대체물로 받지 않는다. 마운트 경로는 folder ID inventory 와 파일 단위로
#    연결됐을 때만 인정된다. runs parent 도 마찬가지다.
#
#    아직 아무것도 열지 않는다. Drive mount 는 승인 후 주석을 푼다.
# from google.colab import drive as _drive
# _drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/MedKOS/ecg-model'
RUNS_PARENT_DIR = f'{DRIVE_ROOT}/runs'

SOURCE_DIR = ''   # 등록 source folder ID 의 마운트 경로 (읽기 전용)
SHARD_DIR  = ''   # 등록 shard folder ID 의 마운트 경로 (읽기 전용)
TARGET_DIR = ''   # 새 corrective 폴더 — RUNS_PARENT_DIR 바로 아래, 아직 없는 이름

print('source folder ID :', R.SOURCE_BUNDLE_FOLDER_ID)
print('shard  folder ID :', R.SHARD_FOLDER_ID)
print('runs parent ID   :', R.RUNS_PARENT_FOLDER_ID)
print('RUNS_PARENT_DIR  :', RUNS_PARENT_DIR)
for _label, _v in (('SOURCE_DIR', SOURCE_DIR), ('SHARD_DIR', SHARD_DIR),
                   ('TARGET_DIR', TARGET_DIR)):
    print(f'{_label:17s}:', _v or '(미지정)')

print()
print('shard 계약 :', R.EXPECTED_SHARD_COUNT, '개 —',
      R.EXPECTED_SHARD_FILENAMES[0], '…', R.EXPECTED_SHARD_FILENAMES[-1])
print('bundle 계약:', len(R.BUNDLE_FILES), '개 · source', len(R.SOURCE_BUNDLE_FILES), '개')
print('NPZ 배열   :', list(R.NPZ_ARRAYS), '· float64', f'({R.N_REPLICATES},)')
print('요청 scope :', R.DRIVE_READONLY_SCOPE)
print('adapter 표면:', R.ADAPTER_OPERATIONS)
print()
print('실행 승인 :', bool(R.EXECUTION_APPROVAL_RECORD.get('granted')))
print(R.APPROVAL_NOTE)

In [ ]:
# 5. 실행 — 승인 전에는 거부된다. 그것이 정상 동작이다.
#
#    인증은 이 셀이 하지 않는다. authenticator 와 service factory 를 넘기면
#    run_repair() 가 guard 아래에서 credential 을 발급하고, 정확히
#    drive.readonly 하나임을 증명한 뒤 service/adapter 를 만든다.
#    승인·token·guard 중 하나라도 막히면 credential 도 service 도 API 호출도
#    0회다.
#
#    route: 승인 → 구현 identity → folder ID → guard → 의존성 → credential →
#           scope 증명 → adapter → runs parent bridge → source snapshot(1회) →
#           shard 자격검증 → 재구성 → null_summary 대조 → NPZ 계약(독립 reader +
#           numpy) → corrective 폴더 조립 → 재검증 → source 재해시 → folder ID 확인
APPROVAL = R.EXECUTION_APPROVAL_TOKEN   # 리터럴을 적지 않는다

DECISION, FAILURE = None, None
try:
    DECISION = R.run_repair(
        SHARD_DIR, SOURCE_DIR, TARGET_DIR, APPROVAL,
        authenticator=R.ColabReadOnlyAuthenticator(),
        service_factory=R.default_service_factory,
        runs_parent_dir=RUNS_PARENT_DIR,
        execution_head=EXECUTION_HEAD,
        repo_root=REPO)
    print('status:', DECISION['status'])
except R.RepairError as _error:
    FAILURE = _error.as_record()
    print('STOP:', FAILURE['first_stopping_reason'])
    print(FAILURE['message'])
    if FAILURE['incomplete_directory']:
        print()
        print('보존된 출력 (삭제하지 않았다):')
        print('  경로 :', FAILURE['incomplete_directory'])
        print('  파일 :', FAILURE['incomplete_listing'])
        print('  상태 :', FAILURE['target_state'],
              '— COMMITTED 아님 · accepted 아님 · 등록 대상 아님')
        print()
        print('folder ID 만 안 잡힌 것이라면 마지막 셀의 reconciliation 을 쓴다.')
        print('그 외에는 새 고유 경로로 재시도한다. 이 폴더를 청소하거나 이어 쓰지 마라.')

    if FAILURE['reconciliation_context']:
        print()
        print('이 record 를 파일로 남겨두면 커널이 죽어도 마지막 셀에서 '
              'folder ID 만 다시 잡을 수 있다:')
        print("  json.dump(FAILURE, open('/content/failure.json','w'), ensure_ascii=False)")


In [ ]:
# 6. 보고 — 이 셀의 저장된 출력이 외부 기록이다(corrective 폴더 안에는
#    provenance 파일을 넣지 않는다). 여기 나온 folder ID 와 digest 를 별도 PR 이
#    Decision log · ASSETS.md · PROJECT_STATE.md 로 옮긴다.
if DECISION is None:
    print('실행되지 않았거나 중단됐다 — 위 셀을 보라. 등록할 값이 없다.')
    if FAILURE:
        print(json.dumps(FAILURE, ensure_ascii=False, indent=2))
else:
    print(R.report_markdown(DECISION))
    print()
    print('--- 복사해 갈 값 ---')
    print('execution head     :', DECISION['pinned_commit'])
    print('approved commit    :',
          DECISION['execution_identity']['approved_implementation_commit'])
    print('NPZ SHA-256        :', DECISION['npz']['sha256'])
    print('corrective folder  :', DECISION['corrective_bundle']['directory'])
    print('corrective folderID:', DECISION['corrective_folder_id']['folder_id'])
    print('scope 증명         :', DECISION['drive_authentication'])
    print('numpy 검증         :', DECISION['npz']['numpy_verification'])
    print()
    print(json.dumps(DECISION['verification']['observed'], indent=2,
                     sort_keys=True))

In [ ]:
# 7. RECONCILIATION (읽기 전용) — 12파일은 다 썼는데 folder ID 만 안 잡혔을 때만.
#    아무것도 쓰지 않고, 두 번째 폴더도 만들지 않는다.
#
#    FAILURE['reconciliation_context'] **하나만** 있으면 된다. DECISION 도,
#    살아 있는 snapshot 변수도 필요 없다 — 그것들을 갖고 있었을 실행이 바로
#    중단된 그 실행이고, 커널을 다시 띄웠으면 둘 다 없다.
#    커널이 죽었다면 저장해 둔 record 를 다시 읽어 CONTEXT 에 넣으면 된다.

CONTEXT = None
if FAILURE and FAILURE['first_stopping_reason'] == R.OUTPUT_FOLDER_ID_UNRESOLVED:
    CONTEXT = FAILURE['reconciliation_context']

# 커널을 다시 띄운 경우: 저장해 둔 failure record 파일 경로를 여기에 적는다.
SAVED_FAILURE_JSON = ''
if CONTEXT is None and SAVED_FAILURE_JSON:
    with open(SAVED_FAILURE_JSON, encoding='utf-8') as _fh:
        CONTEXT = json.load(_fh)['reconciliation_context']

RECONCILED = None
if CONTEXT:
    print('보존된 출력 :', CONTEXT['preserved_directory'])
    print('예상 파일   :', len(CONTEXT['expected_listing']), '개')
    print('NPZ digest  :', CONTEXT['npz_sha256'])
    print('parent ID   :', CONTEXT['runs_parent_folder_id'])
    _adapter, _audit = R.build_drive_adapter(
        APPROVAL, R.ColabReadOnlyAuthenticator(), R.default_service_factory)
    try:
        RECONCILED = R.reconcile_output_folder_id(_adapter, CONTEXT, APPROVAL)
        print()
        print('folder ID 재확인 성공:', RECONCILED['folder_id'])
        print('쓴 것 없음          :', RECONCILED['wrote_nothing'])
        print('context 만으로 수행 :', RECONCILED['from_context_only'])
        print('파일 목록           :', RECONCILED['listing'])
    except R.RepairError as _error:
        print('여전히 미해결:', _error.reason)
        print(_error)
else:
    print('reconciliation 대상 아님 — 위 STOP 이 '
          'REPAIR_OUTPUT_FOLDER_ID_UNRESOLVED 일 때만 쓴다. '
          '커널을 다시 띄웠다면 SAVED_FAILURE_JSON 에 저장한 record 경로를 적어라.')

## 이 노트북이 하지 않는 것

`auth.authenticate_user()` · `build('drive','v3')` 직접 호출 ·
`run_repair_synthetic_fixture()` 호출 · `detect_r()` · beat join 재실행 ·
null 재실행 · M0~M4 집계 · DS2 per-beat label · V10 probability · association ·
S PR-AUC · 학습 · 기존 Drive 파일 이동·**삭제**·덮어쓰기 · frozen 모듈 수정 ·
12파일 계약 완화 · 값 등록.

**등록은 이 실행의 결과가 아니다.** corrective 폴더가 만들어져도 folder ID·
lineage·NPZ digest 는 별도 등록 PR 로만 들어가고, Q5-E PREP P1/P2 재실행은
그와 또 별개의 사용자 승인을 받는다.